In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/datasets/nhatanhadr/xes3g5m-extended/train_valid_sequences_quelevel.csv
/kaggle/input/datasets/nhatanhadr/xes3g5m-extended/test_sequences.csv
/kaggle/input/datasets/nhatanhadr/xes3g5m-extended/test_quelevel.csv
/kaggle/input/datasets/nhatanhadr/xes3g5m-extended/test_window_sequences_quelevel.csv
/kaggle/input/datasets/nhatanhadr/xes3g5m-extended/test.csv
/kaggle/input/datasets/nhatanhadr/xes3g5m-extended/test_question_window_sequences.csv
/kaggle/input/datasets/nhatanhadr/xes3g5m-extended/test_question_sequences.csv
/kaggle/input/datasets/nhatanhadr/xes3g5m-extended/test_window_sequences.csv
/kaggle/input/datasets/nhatanhadr/xes3g5m-extended/test_sequences_quelevel.csv
/kaggle/input/datasets/nhatanhadr/xes3g5m-extended/train_valid_sequences.csv


In [2]:
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import random
from torch.utils.data import Dataset
from collections import defaultdict
from tqdm import tqdm

# 1. Config

In [3]:
config = {
    "seed": 43,
    "model": "DKT",
    "variant": "C",
    "data_train_valid_path": "/kaggle/input/datasets/nhatanhadr/xes3g5m-extended/train_valid_sequences.csv",
    "data_test_path": "/kaggle/input/datasets/nhatanhadr/xes3g5m-extended/test_question_window_sequences.csv",
    "num_concepts": 865,
    "embed_dim": 100,
    "hidden_dim": 200,
    "dropout": 0.2,
    "lr": 1e-3,
    "batch_size": 4,
    "num_epochs": 100,
    "patience": 10,
}

# 2. DataLoading

In [4]:
df_train_valid = pd.read_csv(config["data_train_valid_path"])
df_test = pd.read_csv(config["data_test_path"], nrows=1)

print(f"Train: {len(df_train_valid)}")
print(f"Test: {len(df_test)}")

Train: 33397
Test: 1


# 3. Dataset and DataLoader

In [5]:
def parse_seq(s):
    return list(map(int, s.strip().split(',')))

class DKTDataset(Dataset):
    def __init__(self, df, fold=None, is_train=True):
        if fold is not None:
            if is_train:
                df = df[df['fold'] != fold]
            else:
                df = df[df['fold'] == fold]
        df = df.reset_index(drop=True)


        self.concepts = [parse_seq(x) for x in df['concepts']]
        self.questions = [parse_seq(x) for x in df['questions']]
        self.uid = df['uid'].tolist()
        self.responses = [parse_seq(x) for x in df['responses']]
        self.selectmasks = [parse_seq(x) for x in df['selectmasks']]
        self.is_repeat = [parse_seq(x) for x in df['is_repeat']]

        # Only in test
        self.qidxs = (
            [parse_seq(x) for x in df['qidxs']]
            if 'qidxs' in df.columns else [None]*len(df)
        )

        self.rest = (
            [parse_seq(x) for x in df['rest']]
            if 'rest' in df.columns else [None]*len(df)
        )

        self.len = len(df)

    def __len__(self):
        return self.len

    def __getitem__(self, i):
        raw_c = torch.tensor(self.concepts[i], dtype=torch.long)
        real = (raw_c >= 0)
        
        c = raw_c.clamp(min=0)
        q = torch.tensor(self.questions[i], dtype=torch.long).clamp(min=0)
        u = torch.tensor(self.uid[i], dtype=torch.long)
        r = torch.tensor(self.responses[i], dtype=torch.long).clamp(min=0)
        m = torch.tensor(self.selectmasks[i], dtype=torch.long)
        rp = torch.tensor(self.is_repeat[i], dtype=torch.long)

        item = {
            "concepts": c,
            "questions": q,
            "uid": u,
            "responses": r,
            "selectmasks": m,
            "is_repeat": rp,
            "real": real
        }

        if self.qidxs[i] is not None:
            item["qidxs"] = torch.tensor(self.qidxs[i], dtype=torch.long)

        if self.rest[i] is not None:
            item["rest"] = torch.tensor(self.rest[i], dtype=torch.long)

        return item

# 4. Model Definition

In [6]:
def group_start(rep):
    """Vị trí đầu của câu hỏi chứa t; rep = cờ 'cùng câu hỏi với vị trí trước'."""
    B, L = rep.shape
    pos = torch.arange(L, device=rep.device).expand(B, L)
    return torch.where(rep == 0, pos, torch.zeros_like(pos)).cummax(dim=1).values

class DKT(nn.Module):
    """A/B/C như Khung 4.1; thêm n_labels và hàm inputs để DKT-ML kế thừa."""
 
    def __init__(self, num_concepts, embed_dim=100, hidden_dim=200, dropout=0.2,
                 n_labels=2):
        super().__init__()
        self.num_concepts = num_concepts
        self.emb = nn.Embedding(n_labels * num_concepts, embed_dim)
        self.lstm = nn.LSTM(embed_dim, hidden_dim, batch_first=True)
        self.dropout = nn.Dropout(dropout)
        self.out = nn.Linear(hidden_dim, num_concepts)
 
    def inputs(self, c, r, rep):
        return c + self.num_concepts * r
 
    def forward(self, c, r, rep, group_by_question):
        h, _ = self.lstm(self.emb(self.inputs(c, r, rep)))
        h = self.dropout(h)
        h_shift = torch.zeros_like(h)
        h_shift[:, 1:] = h[:, :-1]
        if group_by_question:                      # đọc trạng thái trước câu hỏi
            idx = group_start(rep).unsqueeze(-1).expand(-1, -1, h.size(-1))
            h_shift = h_shift.gather(1, idx)
        return self.out(h_shift)

# 5. Compute Metrics

In [7]:
# metrics.py – compute_metrics của nhóm (dkt-*.ipynb, ô 11); chỉ đổi ngưỡng sang >=
import numpy as np
from sklearn.metrics import (accuracy_score, average_precision_score, brier_score_loss,
                             f1_score, mean_squared_error, roc_auc_score)
 
 
def expected_calibration_error(y_true, y_prob, n_bins=10):
    y_true, y_prob = np.asarray(y_true), np.asarray(y_prob)
    bins = np.linspace(0.0, 1.0, n_bins + 1)
    ids = np.clip(np.digitize(y_prob, bins[1:-1], right=True), 0, n_bins - 1)
    ece = 0.0
    for b in range(n_bins):
        m = ids == b
        if m.any():
            ece += m.mean() * abs(y_true[m].mean() - y_prob[m].mean())
    return float(ece)
 
 
def compute_metrics(y_true, y_pred, threshold=0.5, n_bins=10):
    y_true = np.asarray(y_true).astype(int)
    y_pred = np.asarray(y_pred).astype(float)
    y_bin = (y_pred >= threshold).astype(int)
    both = len(np.unique(y_true)) > 1          # như nhóm: tập con chỉ có một lớp
    return {"auc": roc_auc_score(y_true, y_pred) if both else 0.5,
            "acc": accuracy_score(y_true, y_bin),
            "rmse": float(np.sqrt(mean_squared_error(y_true, y_pred))),
            "brier": brier_score_loss(y_true, y_pred),
            "ece": expected_calibration_error(y_true, y_pred, n_bins),
            "auprc_wrong": (average_precision_score(1 - y_true, 1 - y_pred) if both
                            else float((1 - y_true).mean())),
            "f1_wrong": f1_score(1 - y_true, 1 - y_bin, zero_division=0)}


# Overfit test

In [8]:
import torch, torch.nn.functional as F
from torch.utils.data import DataLoader

train_dataset = DKTDataset(df_train_valid, fold=0, is_train=True)
train_loader = DataLoader(train_dataset, batch_size=config["batch_size"], shuffle=False)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

model = DKT(
        num_concepts=config.get('num_concepts', 100),
        embed_dim=config.get('embed_dim', 100),
        hidden_dim=config.get('hidden_dim', 200),
        dropout=config.get('dropout', 0.2)
    ).to(device)

import torch, torch.nn as nn

def overfit_test(model, train_loader, device, steps=300, lr=1e-3, n_seq=4):
    # Lấy đúng 1 batch cố định, cắt còn 4 chuỗi (chỉ lấy 1 lần, không lấy lại)
    batch = next(iter(train_loader))
    concepts   = batch["concepts"][:n_seq].to(device)
    responses  = batch["responses"][:n_seq].to(device)
    selectmasks = batch["selectmasks"][:n_seq].to(device)
    is_repeat  = batch["is_repeat"][:n_seq].to(device)

    # Tắt dropout nhưng vẫn để model.train()
    for m in model.modules():
        if isinstance(m, nn.Dropout):
            m.p = 0.0

    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    criterion = nn.BCEWithLogitsLoss(reduction="sum")
    model.train()

    B, L = concepts.shape
    batch_idx = torch.arange(B, device=device).unsqueeze(1)
    time_idx = torch.arange(L, device=device).unsqueeze(0)
    valid = (selectmasks == 1) & (group_start(is_repeat) > 0)

    for t in range(steps):
        optimizer.zero_grad(set_to_none=True)
        logits = model(concepts, responses, is_repeat, True)
        logits_selected = logits[batch_idx, time_idx, concepts]
        loss = criterion(logits_selected[valid], responses[valid].float()) / valid.sum().clamp(min=1)
        loss.backward()
        optimizer.step()
        if t % 50 == 0 or t == steps - 1:
            print(f"step {t}: loss={loss.item():.4f}")
    return loss.item()   # PASS: < ~0.05

overfit_test(model, train_loader, device)


step 0: loss=0.6969
step 50: loss=0.0635
step 100: loss=0.0127
step 150: loss=0.0051
step 200: loss=0.0029
step 250: loss=0.0019
step 299: loss=0.0013


0.0013482870999723673